# 🔍 LLM Hallucination Detection Pipeline v2

**Pipeline Overview:**
1. **Generate** → `Gemini API` (primary) → `google/flan-t5-base` (fallback if Gemini fails)
2. **Domain Identification** → `facebook/bart-large-mnli` (zero-shot classification)
3. **Reference Retrieval** → Serper API (Google Search, top 10 results)
4. **Sentence-Level Validation** → NLI entailment scoring per sentence
5. **Score & Decision** → ACCEPT / WARN / REJECT with response

---
### 🔑 API Keys Required
- **Gemini API Key** → Free at https://aistudio.google.com/app/apikey
- **Serper API Key** → Free tier (2500 searches) at https://serper.dev

> If Gemini API key is missing or fails, the pipeline automatically falls back to `flan-t5-base` (runs locally, no key needed).


Step 0: Install Dependencies

In [ ]:
# Fix: duckduckgo_search renamed to ddgs; also install google-generativeai for Gemini
!pip install -q transformers torch sentencepiece ddgs requests beautifulsoup4 nltk google-generativeai
print("All packages installed.")

In [ ]:
# Suppress all deprecation warnings (including datetime.utcnow from jupyter_client)
import warnings
import logging
warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)
logging.getLogger('urllib3').setLevel(logging.ERROR)

import os
import re
import torch
import nltk
import requests
from bs4 import BeautifulSoup
from transformers import pipeline, T5ForConditionalGeneration, T5Tokenizer

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

device = 0 if torch.cuda.is_available() else -1
print(f"✅ Device: {'GPU (CUDA)' if device == 0 else 'CPU'}")
print(f"✅ PyTorch: {torch.__version__}")

Step 1: Configure API Keys

In [ ]:
# ══════════════════════════════════
# SET YOUR API KEYS HERE
# ══════════════════════════════════

GEMINI_API_KEY = "" #""          # Get free at: https://aistudio.google.com/app/apikey
SERPER_API_KEY = ""          # Get free at: https://serper.dev (2500 free searches)

# ══ OR load from Colab Secrets (recommended) ══
# from google.colab import userdata
# GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
# SERPER_API_KEY = userdata.get('SERPER_API_KEY')

# Validate
GEMINI_AVAILABLE = bool(GEMINI_API_KEY and GEMINI_API_KEY.strip())
SERPER_AVAILABLE = bool(SERPER_API_KEY and SERPER_API_KEY.strip())

print(f"Gemini API Key : {'✅ Set' if GEMINI_AVAILABLE else '❌ Not set → will use flan-t5-base fallback'}")
print(f"Serper API Key : {'✅ Set' if SERPER_AVAILABLE else '❌ Not set → web search disabled'}")

## 🤖 Step 2: Load Models

In [ ]:
# ── Gemini API client (if key available) ──
gemini_model = None
if GEMINI_AVAILABLE:
    try:
        import google.generativeai as genai
        genai.configure(api_key=GEMINI_API_KEY)
        gemini_model = genai.GenerativeModel('gemini-1.5-flash')  # free tier model
        # Quick connectivity test
        test = gemini_model.generate_content("Say: OK")
        print(f"✅ Gemini API connected (gemini-1.5-flash)")
    except Exception as e:
        print(f"⚠️  Gemini API failed: {e}")
        print("    → Falling back to flan-t5-base")
        gemini_model = None  # Ensure gemini_model is None if it fails.
else:
    print("ℹ️  Gemini key not provided → using flan-t5-base")

# ── flan-t5-base (local fallback) ──
# Load flan-t5-base unconditionally, so it's always available as a fallback
print("Loading google/flan-t5-base (local fallback)...")
flan_tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
flan_model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")
flan_model = flan_model.to("cuda" if torch.cuda.is_available() else "cpu")
print("✅ flan-t5-base loaded")

# ── bart-large-mnli for domain ID + NLI ──
print("Loading facebook/bart-large-mnli...")
nli_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device
)
print("✅ bart-large-mnli loaded")
print()
print(f"🔧 Generator : {'Gemini API (gemini-1.5-flash)' if gemini_model else 'flan-t5-base (local)'}")
print(f"🔧 Search    : {'Serper API (Google Search)' if SERPER_AVAILABLE else 'Disabled (no key)'}")

## ⚙️ Step 3: Define Pipeline Functions

In [ ]:
import warnings
import logging
warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)
logging.getLogger('urllib3').setLevel(logging.ERROR)

import os
import re
import torch
import nltk
import requests
from bs4 import BeautifulSoup
from transformers import pipeline, T5ForConditionalGeneration, T5Tokenizer

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

device = 0 if torch.cuda.is_available() else -1

# ══════════════════════════════════════════════════════════
# STEP 1 — Generate
# ══════════════════════════════════════════════════════════
def generate_with_gemini(prompt: str) -> str:
    response = gemini_model.generate_content(
        f"Answer the following question accurately and in detail:\n{prompt}"
    )
    return response.text.strip()

def generate_with_flan(prompt: str, max_new_tokens: int = 300) -> str:
    input_text = f"Answer the following question accurately and in detail:\n{prompt}"
    inputs = flan_tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(flan_model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = flan_model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            num_beams=4, early_stopping=True, no_repeat_ngram_size=3
        )
    return flan_tokenizer.decode(outputs[0], skip_special_tokens=True)

def generate_response(user_input: str) -> tuple:
    if gemini_model is not None:
        try:
            text = generate_with_gemini(user_input)
            return text, "Gemini API (gemini-1.5-flash)"
        except Exception as e:
            print(f"  ⚠️  Gemini failed: {e} → switching to flan-t5-base")
    if flan_model is not None:
        text = generate_with_flan(user_input)
        return text, "flan-t5-base (local fallback)"
    raise RuntimeError("No generation model available.")


# ══════════════════════════════════════════════════════════
# STEP 2 — Domain Identification
# ══════════════════════════════════════════════════════════
DOMAINS = [
    "science", "medicine and health", "history", "technology", "mathematics",
    "law", "politics", "economics", "geography", "sports",
    "entertainment", "philosophy", "literature", "general knowledge"
]

def identify_domain(query: str) -> str:
    result = nli_classifier(query, candidate_labels=DOMAINS, multi_label=False)
    top_domain = result["labels"][0]
    top_score  = result["scores"][0]
    print(f"  📂 Domain: {top_domain} (confidence: {top_score:.2f})")
    return top_domain


# ══════════════════════════════════════════════════════════
# STEP 3 — Reference Retrieval
# ══════════════════════════════════════════════════════════
def fetch_page_text(url: str, max_chars: int = 2500) -> str:
    try:
        headers = {"User-Agent": "Mozilla/5.0 (compatible; HallucinationBot/1.0)"}
        resp = requests.get(url, headers=headers, timeout=6)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()
        paragraphs = soup.find_all("p")
        text = " ".join(p.get_text(separator=" ", strip=True) for p in paragraphs)
        text = re.sub(r'\s+', ' ', text).strip()
        return text[:max_chars]
    except Exception:
        return ""

def search_serper(query: str, top_k: int = 10) -> list:
    url = "https://google.serper.dev/search"
    headers = {"X-API-KEY": SERPER_API_KEY, "Content-Type": "application/json"}
    payload = {"q": query, "num": top_k}
    resp = requests.post(url, headers=headers, json=payload, timeout=10)
    resp.raise_for_status()
    data = resp.json()
    results = []
    for item in data.get("organic", [])[:top_k]:
        results.append({
            "title": item.get("title", ""),
            "url":   item.get("link", ""),
            "snippet": item.get("snippet", "")
        })
    if "answerBox" in data:
        ab = data["answerBox"]
        snippet = ab.get("answer") or ab.get("snippet") or ""
        if snippet:
            results.insert(0, {"title": "Google Answer Box", "url": "", "snippet": snippet})
    return results

def retrieve_references(query: str, domain: str, top_k: int = 10) -> list:
    search_query = f"{query} {domain}"
    references   = []
    if SERPER_AVAILABLE:
        try:
            raw = search_serper(search_query, top_k=top_k)
            print(f"  🌐 Serper: {len(raw)} results fetched")
            for r in raw:
                body = fetch_page_text(r["url"]) if r["url"] else ""
                references.append({
                    "title": r["title"], "url": r["url"],
                    "snippet": r["snippet"],
                    "body": body if body else r["snippet"]
                })
            return references
        except Exception as e:
            print(f"  ⚠️  Serper failed: {e} → trying ddgs fallback")
    try:
        from ddgs import DDGS
        with DDGS() as ddgs:
            raw = list(ddgs.text(search_query, max_results=top_k))
        print(f"  🌐 DDGS fallback: {len(raw)} results")
        for r in raw:
            body = fetch_page_text(r.get("href", ""))
            references.append({
                "title": r.get("title", ""), "url": r.get("href", ""),
                "snippet": r.get("body", ""),
                "body": body if body else r.get("body", "")
            })
    except Exception as e:
        print(f"  ⚠️  DDGS also failed: {e}")
    return references


# ══════════════════════════════════════════════════════════
# STEP 3b — Real-Time Fact Override
# Extracts the ground-truth answer from references and
# compares it directly against the LLM response.
# This catches stale LLM knowledge (e.g. wrong president).
# ══════════════════════════════════════════════════════════

# Named-entity patterns — people, numbers, dates, places
_ENTITY_RE = re.compile(
    r'\b([A-Z][a-z]+ (?:[A-Z][a-z]+ )?[A-Z][a-z]+|'   # proper names (2-3 words)
    r'[A-Z][a-z]{2,} [A-Z][a-z]{2,}|'                  # 2-word proper names
    r'\d{1,3}(?:,\d{3})*(?:\.\d+)?(?:\s?[a-zA-Z%]+)?|' # numbers + units
    r'\d{1,2} [A-Z][a-z]+ \d{4}|'                       # dates like "2 May 1945"
    r'[A-Z][a-z]+ \d{1,2},? \d{4})\b'                   # dates like "May 2, 1945"
)

def extract_answer_from_references(query: str, references: list) -> str | None:
    """
    Search reference snippets for the most-mentioned factual answer
    to the query. Returns the best candidate string, or None.

    Improvements v2:
    1. Detects "who is/was" queries and prioritises PERSON entities
       extracted from answer-style sentences (X is the Y / Y is X)
    2. Scores entities by co-occurrence + answer-pattern bonus
    3. Falls back to original frequency approach if no person found
    """
    STOP = {'who','what','when','where','which','how','is','was','are','were',
            'the','a','an','of','in','on','at','to','for','did','does','do'}
    key_words = [w.lower() for w in re.findall(r'\b\w{3,}\b', query)
                 if w.lower() not in STOP]
    if not key_words:
        return None

    # ── Detect "who is/was" intent ──
    who_query = bool(re.match(r'\b(who\s+is|who\s+was|who\s+are)\b', query, re.I))

    # Collect sentences from top references
    sentences = []
    for ref in references[:8]:
        text = ref.get("body") or ref.get("snippet", "")
        if text:
            for sent in nltk.sent_tokenize(text[:2500]):
                sentences.append(sent)

    # ── Answer-pattern regex for person names ──
    # Matches: "Donald Trump is the president" / "president … Donald Trump"
    ANSWER_PATTERNS = [
        re.compile(
            r'([A-Z][a-z]+(?: [A-Z][a-z.]+){1,3})'   # Name
            r'(?:\s+(?:is|was|has been|became|serves?|assumed))\s+'
            r'(?:the\s+)?(?:current\s+)?'
            r'(?:[A-Za-z ]{0,40})?'
            r'(?:president|prime minister|chancellor|ceo|director|head|leader)',
            re.I
        ),
        re.compile(
            r'(?:president|prime minister|chancellor|ceo|director|head|leader)'
            r'(?:[A-Za-z ,]{0,30})?(?:is|was|:)\s*'
            r'([A-Z][a-z]+(?: [A-Z][a-z.]+){1,3})',
            re.I
        ),
    ]

    # Score candidates
    scores: dict[str, float] = {}

    for sent in sentences:
        sent_lower = sent.lower()
        kw_hits = sum(1 for kw in key_words if kw in sent_lower)
        if kw_hits < 1:
            continue

        # ── Answer-pattern bonus for who-queries ──
        if who_query:
            for pat in ANSWER_PATTERNS:
                for m in pat.finditer(sent):
                    # grab whichever group captured the name
                    name = next((g for g in m.groups() if g and
                                 re.match(r'[A-Z][a-z]', g)), None)
                    if name and len(name) > 4:
                        scores[name] = scores.get(name, 0) + kw_hits * 3  # big bonus
                        break

        # ── General entity scoring ──
        if not who_query or kw_hits >= 2:
            for match in _ENTITY_RE.finditer(sent):
                entity = match.group(0).strip()
                if len(entity) < 5:
                    continue
                # Skip overly generic geographic terms for who-queries
                if who_query and entity.lower() in (
                    'united states', 'america', 'washington', 'congress',
                    'white house', 'oval office'):
                    continue
                scores[entity] = scores.get(entity, 0) + kw_hits

    if not scores:
        return None

    best = max(scores, key=scores.__getitem__)
    return best


def realtime_fact_check(query: str, llm_response: str, references: list) -> dict:
    """
    Compare LLM response against the ground-truth answer extracted
    from live references.

    Returns a dict:
      {
        'override':      bool   — True if we found a conflict,
        'ref_answer':    str    — what references say,
        'conflict_note': str    — human-readable explanation,
        'penalty':       float  — extra contradiction penalty (0.0–0.5)
      }
    """
    ref_answer = extract_answer_from_references(query, references)
    if ref_answer is None:
        return {"override": False, "ref_answer": None,
                "conflict_note": "", "penalty": 0.0}

    resp_lower    = llm_response.lower()
    ref_lower     = ref_answer.lower()
    ref_words     = set(re.findall(r'\b\w{3,}\b', ref_lower))
    resp_words    = set(re.findall(r'\b\w{3,}\b', resp_lower))

    # Overlap ratio between reference answer words and LLM response
    if not ref_words:
        return {"override": False, "ref_answer": ref_answer,
                "conflict_note": "", "penalty": 0.0}

    # For "who is/was" queries use a stricter threshold — generic geo terms
    # like "United States" will otherwise falsely appear in Biden responses
    who_query = bool(re.match(r'\b(who\s+is|who\s+was|who\s+are)\b', query, re.I))
    threshold = 0.55 if who_query else 0.35

    overlap = len(ref_words & resp_words) / len(ref_words)

    if overlap < threshold:
        # Low overlap → likely conflict
        note = (
            f"⚠️  Real-time fact-check conflict detected!\n"
            f"   LLM answer  : {llm_response[:120].strip()}\n"
            f"   References  : '{ref_answer}' (most cited in search results)\n"
            f"   Overlap     : {overlap:.0%} — response does not match current sources."
        )
        print(f"\n  🔴 REAL-TIME FACT-CHECK: {note}")
        return {"override": True, "ref_answer": ref_answer,
                "conflict_note": note, "penalty": 0.45}

    return {"override": False, "ref_answer": ref_answer,
            "conflict_note": "", "penalty": 0.0}


# ══════════════════════════════════════════════════════════
# STEP 4 — Sentence-Level NLI Validation
# ══════════════════════════════════════════════════════════

SHORT_RESPONSE_MIN_WORDS = 6

def build_factual_claim(response: str, query: str) -> str:
    """
    Convert a short answer into a checkable factual claim.
    e.g. query='Who invented the telephone?', response='Alexander Graham Bell'
         → 'Alexander Graham Bell invented the telephone'
    """
    q = query.strip().rstrip("?").strip()
    q_clean = re.sub(
        r'^(who is|who was|who are|what is|what was|what are|when is|when was|'
        r'where is|where was|which is|which was|how is|how was|who)\s+',
        "", q, flags=re.IGNORECASE
    ).strip()
    return f"{response.strip()} is {q_clean}"

def split_sentences(text: str) -> list:
    return [s.strip() for s in nltk.sent_tokenize(text) if s.strip()]

def build_reference_corpus(references: list, max_chars_per_ref: int = 1500) -> str:
    parts = []
    for ref in references:
        text = ref.get("body") or ref.get("snippet", "")
        if text:
            parts.append(text[:max_chars_per_ref])
    return " ".join(parts)

def validate_sentence_nli(sentence: str, reference_corpus: str) -> dict:
    corpus_snippet = reference_corpus[:1500]
    # Escape curly braces in the sentence to prevent them from being interpreted as format specifiers
    escaped_sentence = sentence.replace('{', '{{').replace('}', '}}')
    result = nli_classifier(
        corpus_snippet,
        candidate_labels=["entailment", "neutral", "contradiction"],
        hypothesis_template=f"The relationship of this text to the claim '{escaped_sentence}' is {{}}."
    )
    return dict(zip(result["labels"], result["scores"]))

def validate_all_sentences(response: str, references: list, query: str = "") -> list:
    corpus = build_reference_corpus(references)
    words  = response.strip().split()
    is_short = len(words) < SHORT_RESPONSE_MIN_WORDS

    if is_short and query:
        claim     = build_factual_claim(response, query)
        print(f"  💡 Short response ({len(words)} word(s)) — building factual claim for NLI")
        print(f"     Response : '{response.strip()}'")
        print(f"     Claim    : '{claim}'")
        sentences = [claim]
    else:
        sentences = split_sentences(response)

    if not sentences:
        print("  ⚠️  No sentences to validate")
        return [{"sentence": response,
                 "scores": {"entailment": 0.33, "neutral": 0.34, "contradiction": 0.33}}]

    if not corpus.strip():
        print("  ⚠️  No reference corpus available — skipping NLI validation")
        return [{"sentence": s,
                 "scores": {"entailment": 0.33, "neutral": 0.34, "contradiction": 0.33}}
                for s in sentences]

    print(f"  🔬 Validating {len(sentences)} sentence(s)...")
    results = []
    for i, sent in enumerate(sentences):
        scores       = validate_sentence_nli(sent, corpus)
        display_text = response if is_short else sent
        results.append({"sentence": display_text, "scores": scores})
        e = scores.get("entailment", 0)
        n = scores.get("neutral", 0)
        c = scores.get("contradiction", 0)
        flag = "✅" if e > 0.5 else ("❌" if c > 0.45 else "⚠️")
        print(f"    [{flag}] S{i+1}: E={e:.2f} | N={n:.2f} | C={c:.2f}")
        print(f"         Checking claim: '{sent[:90]}{'...' if len(sent) > 90 else ''}'")
    return results


# ══════════════════════════════════════════════════════════
# STEP 5 — Scoring & Decision
# ══════════════════════════════════════════════════════════n
ACCEPT_ENTAILMENT_MIN    = 0.45
ACCEPT_CONTRADICTION_MAX = 0.30
REJECT_CONTRADICTION_MIN = 0.40  # lowered: fact-check penalty of 0.45 should always trigger REJECT

def compute_score(validation_results: list, fact_check: dict = None) -> dict:
    """
    Aggregate NLI scores + apply real-time fact-check penalty if triggered.
    The penalty directly boosts the contradiction score to force a REJECT
    when live references contradict the LLM response.
    """
    if not validation_results:
        return {"entailment": 0.33, "neutral": 0.34, "contradiction": 0.33,
                "overall": 0.50, "total_sentences": 0}

    n     = len(validation_results)
    avg_e = sum(r["scores"].get("entailment", 0)    for r in validation_results) / n
    avg_n = sum(r["scores"].get("neutral", 0)        for r in validation_results) / n
    avg_c = sum(r["scores"].get("contradiction", 0)  for r in validation_results) / n

    # ── Apply real-time fact-check penalty ──
    penalty = fact_check.get("penalty", 0.0) if fact_check else 0.0
    if penalty > 0:
        print(f"  🔴 Applying real-time fact-check penalty: +{penalty:.2f} to contradiction")
        avg_c = min(1.0, avg_c + penalty)
        avg_e = max(0.0, avg_e - penalty * 0.5)

    overall = round((avg_e - avg_c + 1) / 2, 4)
    return {"entailment": avg_e, "neutral": avg_n,
            "contradiction": avg_c, "overall": overall,
            "total_sentences": n, "fact_check_penalty": penalty}


def make_decision(score: dict, llm_response: str, references: list,
                  model_used: str, fact_check: dict = None) -> dict:
    e        = score["entailment"]
    c        = score["contradiction"]
    overall  = score["overall"]
    ref_urls = [r["url"] for r in references[:3] if r.get("url")]
    penalty  = score.get("fact_check_penalty", 0.0)

    # Build conflict note for display
    conflict_note = ""
    if fact_check and fact_check.get("override"):
        conflict_note = (
            f"\n🔴 Real-Time Fact-Check Override:\n"
            f"   References say: '{fact_check['ref_answer']}'\n"
            f"   LLM said      : '{llm_response[:80].strip()}...'\n"
            f"   This mismatch triggered a contradiction penalty of +{penalty:.2f}."
        )

    if e >= ACCEPT_ENTAILMENT_MIN and c < ACCEPT_CONTRADICTION_MAX:
        verdict    = "✅ ACCEPT"
        assessment = (
            f"Response is well-supported by online references.\n"
            f"Factuality Score: {overall:.3f} / 1.000  |  "
            f"Entailment: {e:.2f}  Contradiction: {c:.2f}"
        )
        final_response = llm_response

    elif c >= REJECT_CONTRADICTION_MIN:
        verdict    = "❌ REJECT"
        assessment = (
            f"Response contains significant factual inconsistencies.\n"
            f"Factuality Score: {overall:.3f} / 1.000  |  "
            f"Entailment: {e:.2f}  Contradiction: {c:.2f}\n"
            f"⚠️  Likely hallucination — do NOT rely on this response."
            f"{conflict_note}"
        )
        final_response = (
            f"[❌ HALLUCINATION DETECTED — Response Rejected]\n"
            f"─────────────────────────────────────────────\n"
            f"{llm_response}\n"
            f"─────────────────────────────────────────────\n"
            f"⚠️  The above output was flagged as potentially inaccurate.\n"
            f"Please verify with authoritative sources."
            + (f"\n{conflict_note}" if conflict_note else "")
        )
    else:
        verdict    = "⚠️  WARN"
        assessment = (
            f"Response is partially supported but contains uncertain claims.\n"
            f"Factuality Score: {overall:.3f} / 1.000  |  "
            f"Entailment: {e:.2f}  Contradiction: {c:.2f}\n"
            f"Treat with caution and verify key facts."
            f"{conflict_note}"
        )
        final_response = (
            f"[⚠️  CAUTION: Partial verification — treat with care]\n"
            f"─────────────────────────────────────────────\n"
            f"{llm_response}"
            + (f"\n{conflict_note}" if conflict_note else "")
        )

    return {
        "verdict": verdict, "assessment": assessment,
        "final_response": final_response, "score": score,
        "model_used": model_used, "reference_urls": ref_urls
    }

print("✅ Pipeline functions loaded (v5 — real-time fact-check override).")

## 🚀 Step 4: Main Pipeline Runner

In [ ]:
def run_pipeline(user_input: str) -> dict:
    """
    Full hallucination detection pipeline.

    Args:
        user_input: User query/question
    Returns:
        dict with all pipeline outputs
    """
    SEP = "═" * 62
    print(SEP)
    print(f"📝 Query : {user_input}")
    print(SEP)

    # ── STEP 1: Generate ──
    print("\n[Step 1] 🤖 Generating response...")
    llm_response, model_used = generate_response(user_input)
    print(f"  Model   : {model_used}")
    print(f"  Response: {llm_response[:250]}{'...' if len(llm_response) > 250 else ''}")

    # ── STEP 2: Domain Identification ──
    print("\n[Step 2] 🏷️  Identifying domain...")
    domain = identify_domain(user_input)

    # ── STEP 3: Reference Retrieval ──
    print("\n[Step 3] 🌐 Retrieving references...")
    references = retrieve_references(user_input, domain, top_k=10)
    print(f"  Total references with body text: "
          f"{sum(1 for r in references if r.get('body'))}")

    # ── STEP 4: Sentence Validation ──
    print("\n[Step 4] 🔬 Sentence-level NLI validation...")
    validation_results = validate_all_sentences(llm_response, references, query=user_input)

    # ── STEP 4b: Real-Time Fact Check ──
    print("\n[Step 4b] 🔴 Real-time fact-check against references...")
    fact_check = realtime_fact_check(user_input, llm_response, references)
    if fact_check['override']:
        print(f"  ❌ Conflict found! References point to: '{fact_check['ref_answer']}'")
    else:
        print(f"  ✅ No conflict detected. Ref answer: {fact_check['ref_answer']}")

    # ── STEP 5: Score & Decision ──
    print("\n[Step 5] 📊 Scoring & Decision...")
    score = compute_score(validation_results, fact_check=fact_check)
    decision = make_decision(score, llm_response, references, model_used, fact_check=fact_check)

    # ── Print Final Result ──
    print("\n" + SEP)
    print("  PIPELINE RESULT")
    print(SEP)
    print(f"  Verdict        : {decision['verdict']}")
    print(f"  Factuality     : {score['overall']:.3f} / 1.000")
    print(f"  Entailment     : {score['entailment']:.3f}")
    print(f"  Neutral        : {score['neutral']:.3f}")
    print(f"  Contradiction  : {score['contradiction']:.3f}")
    print(f"  Sentences      : {score['total_sentences']}")
    print(f"  Generator      : {model_used}")
    print()
    print("📌 Assessment:")
    print(f"   {decision['assessment']}")
    if decision['reference_urls']:
        print("\n🔗 Top Sources:")
        for url in decision['reference_urls']:
            print(f"   • {url}")
    print("\n📄 Final Response:")
    print(decision['final_response'])
    print(SEP)

    return {
        "query": user_input,
        "llm_response": llm_response,
        "model_used": model_used,
        "domain": domain,
        "verdict": decision["verdict"],
        "score": score,
        "final_response": decision["final_response"],
        "references": references,
        "validation_results": validation_results
    }

print("✅ Pipeline runner ready.")

## ▶️ Step 5: Run on a Single Query

In [ ]:
USER_QUERY = "Who invented the telephone?"

result = run_pipeline(USER_QUERY)

## 🧪 Step 6: Batch Test

In [ ]:
test_queries = [
    "Is paracetomal used to cure dengue ?"

]

all_results = []
for q in test_queries:
    r = run_pipeline(q)
    all_results.append(r)
    print()

In [ ]:
# ── Summary Table ──
print("\n📊 BATCH SUMMARY")
print("─" * 85)
print(f"{'Query':<42} {'Domain':<16} {'Score':>6}  {'Verdict'}")
print("─" * 85)
for r in all_results:
    q = (r['query'][:40] + '..') if len(r['query']) > 40 else r['query']
    d = (r['domain'][:14] + '..') if len(r['domain']) > 14 else r['domain']
    print(f"{q:<42} {d:<16} {r['score']['overall']:>6.3f}  {r['verdict']}")
print("─" * 85)

## 📋 Step 6b: Batch Results Table

In [ ]:
# ════════════════════════════════════════════════════════════════
# 📋 METRICS SUMMARY TABLE — Per-Query + Aggregate
# ════════════════════════════════════════════════════════════════

def print_metrics_table(results: list):
    """
    Prints a formatted table with per-query NLI scores, verdict,
    hallucination flag, and token count estimate.
    Also prints aggregate detection metrics at the bottom.
    """
    COL = {
        'query':    42,
        'domain':   18,
        'e':         8,
        'n':         8,
        'c':         8,
        'score':     8,
        'tokens':    8,
        'verdict':  12,
    }

    # ── Header ──
    SEP = "═" * 130
    sep = "─" * 130
    print("\n" + SEP)
    print("  📋  BATCH RESULTS — DETAILED METRICS TABLE")
    print(SEP)
    print(
        f"  {'Query':<{COL['query']}} "
        f"{'Domain':<{COL['domain']}} "
        f"{'Entail':>{COL['e']}} "
        f"{'Neutral':>{COL['n']}} "
        f"{'Contra':>{COL['c']}} "
        f"{'Score':>{COL['score']}} "
        f"{'Tokens':>{COL['tokens']}} "
        f"{'Verdict':<{COL['verdict']}}"
    )
    print(sep)

    # ── Per-row ──
    total_e, total_n, total_c, total_score = 0, 0, 0, 0
    y_pred, y_true = [], []
    total_tokens = 0

    for r in results:
        s       = r['score']
        verdict = r['verdict']
        q       = (r['query'][:40] + '..') if len(r['query']) > 40 else r['query']
        d       = (r['domain'][:16] + '..') if len(r['domain']) > 16 else r['domain']

        # Token estimate
        q_tok   = int(len(r['query'].split()) * 1.3)
        r_tok   = int(len(r['llm_response'].split()) * 1.3)
        ref_tok = sum(int(len((ref.get('body') or ref.get('snippet','')).split()) * 1.3)
                      for ref in r.get('references', []))
        tokens  = q_tok + r_tok + ref_tok
        total_tokens += tokens

        # Binary labels for detection metrics
        pred = 1 if 'REJECT' in verdict else 0
        true = 1 if s['contradiction'] > 0.45 else 0
        y_pred.append(pred); y_true.append(true)

        # Verdict icon only (shorter)
        icon = '✅ ACCEPT' if 'ACCEPT' in verdict else ('❌ REJECT' if 'REJECT' in verdict else '⚠️  WARN')

        print(
            f"  {q:<{COL['query']}} "
            f"{d:<{COL['domain']}} "
            f"{s['entailment']:>{COL['e']}.3f} "
            f"{s['neutral']:>{COL['n']}.3f} "
            f"{s['contradiction']:>{COL['c']}.3f} "
            f"{s['overall']:>{COL['score']}.3f} "
            f"{tokens:>{COL['tokens']},} "
            f"{icon:<{COL['verdict']}}"
        )

        total_e     += s['entailment']
        total_n     += s['neutral']
        total_c     += s['contradiction']
        total_score += s['overall']

    # ── Averages row ──
    n = len(results)
    print(sep)
    print(
        f"  {'AVERAGE':<{COL['query']}} "
        f"{'':<{COL['domain']}} "
        f"{total_e/n:>{COL['e']}.3f} "
        f"{total_n/n:>{COL['n']}.3f} "
        f"{total_c/n:>{COL['c']}.3f} "
        f"{total_score/n:>{COL['score']}.3f} "
        f"{total_tokens//n:>{COL['tokens']},} "
    )
    print(SEP)

    # ── Detection Metrics ──
    tp = sum(1 for p,t in zip(y_pred,y_true) if p==1 and t==1)
    fp = sum(1 for p,t in zip(y_pred,y_true) if p==1 and t==0)
    fn = sum(1 for p,t in zip(y_pred,y_true) if p==0 and t==1)
    tn = sum(1 for p,t in zip(y_pred,y_true) if p==0 and t==0)
    precision = tp/(tp+fp) if (tp+fp)>0 else 0.0
    recall    = tp/(tp+fn) if (tp+fn)>0 else 0.0
    f1        = 2*precision*recall/(precision+recall) if (precision+recall)>0 else 0.0
    hallu_rate= sum(1 for t in y_true if t==1)/n
    hi        = (total_c/n + 0.5*(total_n/n)) / 3
    ter       = total_tokens / max(sum(int(len(r['query'].split())*1.3) for r in results), 1)

    print("\n  📊  AGGREGATE DETECTION METRICS")
    print(sep)
    print(f"  {'Metric':<30} {'Value':>10}   {'Description'}")
    print(sep)
    rows = [
        ("Total Queries",          f"{n}",            "Queries processed"),
        ("TP / FP / FN / TN",      f"{tp}/{fp}/{fn}/{tn}", "Confusion matrix values"),
        ("Precision",              f"{precision:.4f}",  "TP / (TP+FP)"),
        ("Recall",                 f"{recall:.4f}",     "TP / (TP+FN)"),
        ("F1 Score",               f"{f1:.4f}",         "Harmonic mean of P & R"),
        ("Hallucination Rate",     f"{hallu_rate:.4f}", "Fraction of queries flagged"),
        ("Hallucination Index (HI)",f"{hi:.4f}",        "(Contra + 0.5×Neutral) / 3"),
        ("Token Expansion Ratio",  f"{ter:.2f}x",       "Total tokens / Input tokens"),
        ("Avg Entailment",         f"{total_e/n:.4f}",  "Higher = better supported"),
        ("Avg Neutral",            f"{total_n/n:.4f}",  "Uncertain / unverifiable"),
        ("Avg Contradiction",      f"{total_c/n:.4f}",  "Lower = fewer conflicts"),
        ("Avg Factuality Score",   f"{total_score/n:.4f}", "Overall accuracy (0–1)"),
    ]
    for label, val, desc in rows:
        print(f"  {label:<30} {val:>10}   {desc}")
    print(SEP + "\n")


# ── Run it ──
print_metrics_table(all_results)


## 🔧 Configuration & Thresholds

| Parameter | Default | Description |
|---|---|---|
| `ACCEPT_ENTAILMENT_MIN` | 0.50 | Avg entailment must meet this to ACCEPT |
| `ACCEPT_CONTRADICTION_MAX` | 0.25 | Avg contradiction must be below this to ACCEPT |
| `REJECT_CONTRADICTION_MIN` | 0.40 | Avg contradiction above this → REJECT |
| `top_k` | 10 | Serper search results to fetch |
| `max_chars_per_ref` | 1500 | Max chars used per reference in NLI corpus |

### Models
| Model | Size | Use |
|---|---|---|
| `gemini-1.5-flash` (API) | Cloud | Primary generator — free tier |
| `google/flan-t5-base` | ~250 MB | Local fallback generator |
| `facebook/bart-large-mnli` | ~1.6 GB | Domain ID + NLI validation |

### Verdict Logic
```
entailment ≥ 0.50 AND contradiction < 0.25  →  ✅ ACCEPT
contradiction ≥ 0.40                         →  ❌ REJECT
everything else                              →  ⚠️  WARN
```

### Search Priority
```
1. Serper API (Google Search) — best quality, needs free key
2. ddgs (DuckDuckGo)          — no key needed, automatic fallback
```

## 📊 Step 7: Analytics & Performance Evaluation

This section computes all evaluation metrics from the base paper:
- **Detection Scores**: Precision, Recall, F1
- **NLI Distribution**: Entailment / Neutral / Contradiction
- **Hallucination Rate & Index (HI)**
- **Token Expansion Ratio (TER)**
- **Visualizations**: Bar Charts, Radar Chart, Confusion Matrix Heatmap, Pie Chart, Line Chart (Ablation)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Try to import sklearn for confusion matrix ──
try:
    from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False
    print("⚠️  sklearn not found. Run: pip install scikit-learn --break-system-packages")

plt.rcParams.update(
    {
        'figure.facecolor': '#ffffff',  # White background for the figure
        'axes.facecolor': '#ffffff',  # White background for the axes
        'axes.edgecolor': '#cccccc',  # Light gray edge for axes
        'axes.labelcolor': '#333333',  # Darker label color
        'xtick.color': '#333333',  # Darker x-tick color
        'ytick.color': '#333333',  # Darker y-tick color
        'text.color': '#333333',  # Darker text color
        'grid.color': '#e0e0e0',  # Light gray grid lines
        'axes.titlecolor': '#1f77b4',  # Blue for titles (default matplotlib blue)
        'font.family': 'DejaVu Sans',
    }
)
ACCENT = ['#58a6ff', '#3fb950', '#f78166', '#d2a8ff', '#ffa657', '#79c0ff']

print("✅ Analytics module loaded.")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 1️⃣  EXTRACT METRICS FROM BATCH RESULTS
# ════════════════════════════════════════════════════════════════

def extract_metrics(results: list) -> dict:
    """
    Derive detection + NLI metrics from all_results.
    Verdict → binary label: REJECT=hallucination(1), else(0)
    We treat high contradiction as the 'ground truth' proxy.
    """
    y_pred, y_true = [], []
    entailments, neutrals, contradictions = [], [], []
    overall_scores, queries, verdicts = [], [], []
    total_input_tokens, total_all_tokens = 0, 0

    for r in results:
        s = r['score']
        verdict = r['verdict']

        # Binary prediction: REJECT → predicted hallucination (1)
        pred = 1 if 'REJECT' in verdict else 0
        # Proxy ground truth: contradiction dominant → actual hallucination
        true = 1 if s['contradiction'] > 0.40 else 0

        y_pred.append(pred)
        y_true.append(true)
        entailments.append(s['entailment'])
        neutrals.append(s['neutral'])
        contradictions.append(s['contradiction'])
        overall_scores.append(s['overall'])
        queries.append(r['query'][:35] + ('...' if len(r['query']) > 35 else ''))
        verdicts.append(verdict)

        # TER: estimate tokens (approx word count * 1.3)
        q_tokens  = int(len(r['query'].split()) * 1.3)
        resp_tokens = int(len(r['llm_response'].split()) * 1.3)
        ref_tokens  = sum(int(len((ref.get('body') or ref.get('snippet','')).split()) * 1.3)
                          for ref in r.get('references', []))
        total_input_tokens += q_tokens
        total_all_tokens   += q_tokens + resp_tokens + ref_tokens

    # ── Compute detection metrics ──
    tp = sum(1 for p, t in zip(y_pred, y_true) if p == 1 and t == 1)
    fp = sum(1 for p, t in zip(y_pred, y_true) if p == 1 and t == 0)
    fn = sum(1 for p, t in zip(y_pred, y_true) if p == 0 and t == 1)
    tn = sum(1 for p, t in zip(y_pred, y_true) if p == 0 and t == 0)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)

    # ── NLI Distribution ──
    n = len(results)
    avg_e = np.mean(entailments)
    avg_n = np.mean(neutrals)
    avg_c = np.mean(contradictions)

    hallucination_rate = sum(1 for c in contradictions if c > 0.4) / n
    hi = (sum(contradictions) + 0.5 * sum(neutrals)) / (n * 3)  # normalised HI

    ter = total_all_tokens / max(total_input_tokens, 1)

    print("══════════════════════════════════════════════")
    print("  📊 EVALUATION METRICS SUMMARY")
    print("══════════════════════════════════════════════")
    print(f"  Total Queries      : {n}")
    print(f"  TP={tp}  FP={fp}  FN={fn}  TN={tn}")
    print(f"  Precision          : {precision:.3f}")
    print(f"  Recall             : {recall:.3f}")
    print(f"  F1 Score           : {f1:.3f}")
    print(f"  Hallucination Rate : {hallucination_rate:.3f}")
    print(f"  Hallucination Index: {hi:.4f}")
    print(f"  Token Exp. Ratio   : {ter:.2f}x")
    print(f"  Avg Entailment     : {avg_e:.3f}")
    print(f"  Avg Neutral        : {avg_n:.3f}")
    print(f"  Avg Contradiction  : {avg_c:.3f}")
    print("══════════════════════════════════════════════")

    return dict(
        y_pred=y_pred, y_true=y_true,
        tp=tp, fp=fp, fn=fn, tn=tn,
        precision=precision, recall=recall, f1=f1,
        hallucination_rate=hallucination_rate, hi=hi, ter=ter,
        avg_e=avg_e, avg_n=avg_n, avg_c=avg_c,
        entailments=entailments, neutrals=neutrals,
        contradictions=contradictions, overall_scores=overall_scores,
        queries=queries, verdicts=verdicts, n=n
    )


# ── Run metrics extraction ──
metrics = extract_metrics(all_results)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# -- Try to import sklearn for confusion matrix --
try:
    from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False
    print("⚠️  sklearn not found. Run: pip install scikit-learn --break-system-packages")

plt.rcParams.update(
    {
        'figure.facecolor': '#ffffff',  # White background for the figure
        'axes.facecolor': '#ffffff',  # White background for the axes
        'axes.edgecolor': '#cccccc',  # Light gray edge for axes
        'axes.labelcolor': '#333333',  # Darker label color
        'xtick.color': '#333333',  # Darker x-tick color
        'ytick.color': '#333333',  # Darker y-tick color
        'text.color': '#333333',  # Darker text color
        'grid.color': '#e0e0e0',  # Light gray grid lines
        'axes.titlecolor': '#1f77b4',  # Blue for titles (default matplotlib blue)
        'font.family': 'DejaVu Sans',
    }
)
ACCENT = ['#58a6ff', '#3fb950', '#f78166', '#d2a8ff', '#ffa657', '#79c0ff']

print("✅ Analytics module loaded.")

# -------------------------------------------------------
# 2. ALL VISUALIZATIONS (6 plots in one figure)
# ---------------------------------------------------------

def plot_all_analytics(m: dict):
    fig = plt.figure(figsize=(22, 18))
    fig.patch.set_facecolor('#ffffff')
    fig.suptitle('Hallucination Detection Pipeline -- Analytics Dashboard',
                 fontsize=17, fontweight='bold', color='#1f77b4', y=0.98)

    # -- Layout: 3 rows x 2 cols + bottom row (1 wide + 1) --
    gs = fig.add_gridspec(3, 2, hspace=0.45, wspace=0.38)

    # ---------------------------------------------------------
    # A) Detection Performance Bar Chart (Precision / Recall / F1)
    # ---------------------------------------------------------
    ax1 = fig.add_subplot(gs[0, 0])
    det_labels  = ['Precision', 'Recall', 'F1 Score']
    det_vals    = [m['precision'], m['recall'], m['f1']]
    bar_colors  = [ACCENT[0], ACCENT[1], ACCENT[3]]
    bars = ax1.bar(det_labels, det_vals, color=bar_colors,
                   edgecolor='#30363d', linewidth=0.8, width=0.55)
    for bar, val in zip(bars, det_vals):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.3f}', ha='center', va='bottom',
                 fontsize=11, fontweight='bold', color='#333333')
    ax1.set_ylim(0, 1.15)
    ax1.set_title('Detection Performance', fontsize=13, pad=10)
    ax1.set_ylabel('Score')
    ax1.axhline(0.5, color='#f78166', linestyle='--', linewidth=0.8, alpha=0.7, label='0.5 baseline')
    ax1.legend(fontsize=9)
    ax1.grid(axis='y', alpha=0.3)

    # ---------------------------------------------------------
    # B) NLI Distribution Pie Chart
    # ---------------------------------------------------------
    ax2 = fig.add_subplot(gs[0, 1])
    pie_vals    = [m['avg_e'], m['avg_n'], m['avg_c']]
    pie_labels  = [f'Entailment\n{m["avg_e"]:.3f}',
                   f'Neutral\n{m["avg_n"]:.3f}',
                   f'Contradiction\n{m["avg_c"]:.3f}']
    pie_colors  = [ACCENT[1], ACCENT[4], ACCENT[2]]
    wedges, texts, autotexts = ax2.pie(
        pie_vals, labels=pie_labels, colors=pie_colors,
        autopct='%1.1f%%', startangle=140,
        textprops={'color': '#333333', 'fontsize': 10},
        wedgeprops={'edgecolor': '#ffffff', 'linewidth': 1.5})
    for at in autotexts:
        at.set_color('#333333'); at.set_fontweight('bold')
    ax2.set_title('Avg NLI Score Distribution', fontsize=13, pad=10)

    # ---------------------------------------------------------
    # C) Per-Query Factuality Score (grouped bar: E / N / C)
    # ---------------------------------------------------------
    ax3 = fig.add_subplot(gs[1, :])
    x   = np.arange(m['n'])
    w   = 0.25
    ax3.bar(x - w, m['entailments'],    width=w, label='Entailment',   color=ACCENT[1], edgecolor='#cccccc')
    ax3.bar(x,     m['neutrals'],       width=w, label='Neutral',      color=ACCENT[4], edgecolor='#cccccc')
    ax3.bar(x + w, m['contradictions'], width=w, label='Contradiction', color=ACCENT[2], edgecolor='#cccccc')
    ax3.set_xticks(x)
    ax3.set_xticklabels(m['queries'], rotation=30, ha='right', fontsize=9)
    ax3.set_ylim(0, 1.15)
    ax3.set_ylabel('NLI Score')
    ax3.set_title('Per-Query NLI Breakdown (Entailment / Neutral / Contradiction)', fontsize=13, pad=10)
    ax3.legend(loc='upper right', fontsize=10)
    ax3.grid(axis='y', alpha=0.3)
    # Verdict labels on top
    for i, v in enumerate(m['verdicts']):
        icon = '✅' if 'ACCEPT' in v else ('❌' if 'REJECT' in v else '⚠️')
        ax3.text(i, 1.08, icon, ha='center', fontsize=11)

    # ---------------------------------------------------------
    # D) Confusion Matrix Heatmap
    # ---------------------------------------------------------
    ax4 = fig.add_subplot(gs[2, 0])
    cm = np.array([[m['tn'], m['fp']], [m['fn'], m['tp']]])
    im = ax4.imshow(cm, cmap='Blues', vmin=0)
    for i in range(2):
        for j in range(2):
            ax4.text(j, i, str(cm[i, j]), ha='center', va='center',
                     fontsize=20, fontweight='bold',
                     color='white' if cm[i, j] > cm.max()/2 else '#333333')
    ax4.set_xticks([0, 1]); ax4.set_yticks([0, 1])
    ax4.set_xticklabels(['Pred: Non-Hallu', 'Pred: Hallu'])
    ax4.set_yticklabels(['Actual: Non-Hallu', 'Actual: Hallu'])
    ax4.set_title('Confusion Matrix', fontsize=13, pad=10)
    plt.colorbar(im, ax=ax4)

    # ---------------------------------------------------------
    # E) Radar Chart (multi-metric overview)
    # ---------------------------------------------------------
    ax5 = fig.add_subplot(gs[2, 1], polar=True)
    radar_labels = ['Precision', 'Recall', 'F1',
                    'Avg Entailment', '1-Hallu Rate', '1-TER(norm)']
    ter_norm = max(0, 1 - (m['ter'] - 1) / 20)   # normalise TER: lower=better->higher score
    radar_vals = [
        m['precision'], m['recall'], m['f1'],
        m['avg_e'], 1 - m['hallucination_rate'], ter_norm
    ]
    N = len(radar_labels)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]
    radar_vals_plot = radar_vals + radar_vals[:1]

    ax5.set_facecolor('#ffffff')
    ax5.plot(angles, radar_vals_plot, 'o-', color=ACCENT[0], linewidth=2)
    ax5.fill(angles, radar_vals_plot, alpha=0.25, color=ACCENT[0])
    ax5.set_xticks(angles[:-1])
    ax5.set_xticklabels(radar_labels, size=9, color='#333333')
    ax5.set_ylim(0, 1)
    ax5.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax5.set_yticklabels(['0.25','0.5','0.75','1.0'], size=7, color='#333333')
    ax5.grid(color='#e0e0e0', linewidth=0.6);
    ax5.set_title('Radar: Multi-Metric Overview', fontsize=13, pad=18, color='#1f77b4')
    ax5.spines['polar'].set_color('#cccccc')

    plt.savefig('hallucination_analytics.png',
                dpi=150, bbox_inches='tight',
                facecolor='#ffffff', edgecolor='none')
    plt.show()
    print("✅ Dashboard saved -> hallucination_analytics.png")

# ---------------------------------------------------------
# 3. ABLATION STUDY: Effect of Top-K on NLI Scores
# ---------------------------------------------------------
# Uses the first query in all_results as the probe query.
# We simulate top-k by slicing reference lists.

def run_ablation(probe_result: dict, k_values=(1, 2, 3, 5, 8, 10)):
    print("🔬 Running ablation study (Top-K reference slices)...")
    k_results = []
    refs_full = probe_result['references']
    response  = probe_result['llm_response']

    for k in k_values:
        refs_k = refs_full[:k]
        val_k  = validate_all_sentences(response, refs_k)
        sc_k   = compute_score(val_k)
        k_results.append({'k': k, **sc_k})
        print(f"  k={k:2d}: E={sc_k['entailment']:.3f}  N={sc_k['neutral']:.3f}  "
              f"C={sc_k['contradiction']:.3f}  Overall={sc_k['overall']:.3f}")

    # -- Line Chart --
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor('#ffffff')
    ax.set_facecolor('#ffffff')

    ks = [r['k'] for r in k_results]
    ax.plot(ks, [r['entailment']    for r in k_results], 'o-', color=ACCENT[1], lw=2, label='Entailment')
    ax.plot(ks, [r['neutral']       for r in k_results], 's-', color=ACCENT[4], lw=2, label='Neutral')
    ax.plot(ks, [r['contradiction']  for r in k_results], '^-', color=ACCENT[2], lw=2, label='Contradiction')
    ax.plot(ks, [r['overall']       for r in k_results], 'D-', color=ACCENT[0], lw=2.5, label='Overall Factuality', linestyle='--')

    ax.set_xlabel('Top-K References', fontsize=12)
    ax.set_ylabel('NLI Score', fontsize=12)
    ax.set_title('Ablation Study: Effect of Top-K on NLI Scores', fontsize=14, color='#1f77b4')
    ax.set_xticks(ks)
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)
    ax.set_ylim(0, 1.05)
    for spine in ax.spines.values(): spine.set_edgecolor('#cccccc')

    plt.tight_layout()
    plt.savefig('ablation_study.png',
                dpi=150, bbox_inches='tight',
                facecolor='#ffffff', edgecolor='none')
    plt.show()
    print("✅ Ablation chart saved -> ablation_study.png")
    return k_results

# ---------------------------------------------------------
# 4. TOKEN EXPANSION RATIO (TER) -- Per-Query Bar Chart
# ---------------------------------------------------------

def plot_ter(results: list):
    ters, labels = [], []
    for r in results:
        q_tok    = max(1, int(len(r['query'].split()) * 1.3))
        resp_tok = int(len(r['llm_response'].split()) * 1.3)
        ref_tok  = sum(int(len((ref.get('body') or ref.get('snippet','')).split()) * 1.3)
                       for ref in r.get('references', []))
        ter = round((q_tok + resp_tok + ref_tok) / q_tok, 2)
        ters.append(ter)
        labels.append(r['query'][:30] + ('...' if len(r['query']) > 30 else ''))

    fig, ax = plt.subplots(figsize=(12, 5))
    fig.patch.set_facecolor('#ffffff')
    ax.set_facecolor('#ffffff')

    bar_cols = [ACCENT[2] if t > np.mean(ters) else ACCENT[1] for t in ters]
    bars = ax.bar(range(len(ters)), ters, color=bar_cols,
                  edgecolor='#cccccc', linewidth=0.8)
    for bar, val in zip(bars, ters):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}x', ha='center', fontsize=9, color='#333333')

    ax.axhline(np.mean(ters), color='#ffa657', linestyle='--', linewidth=1.2,
               label=f'Mean TER = {np.mean(ters):.1f}x')
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Token Expansion Ratio (TER)', fontsize=11)
    ax.set_title('TER per Query  (lower = more efficient)', fontsize=14, color='#1f77b4')
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    for spine in ax.spines.values(): spine.set_edgecolor('#cccccc')

    legend_patches = [
        mpatches.Patch(color=ACCENT[2], label='Above-average TER'),
        mpatches.Patch(color=ACCENT[1], label='Below-average TER')
    ]
    ax.legend(handles=legend_patches + [ax.get_legend_handles_labels()[0][-1]],
              fontsize=9)

    plt.tight_layout()
    plt.savefig('ter_chart.png',
                dpi=150, bbox_inches='tight',
                facecolor='#ffffff', edgecolor='none')
    plt.show()
    print("✅ TER chart saved -> ter_chart.png")


# Consolidate execution

test_queries = [
    "Is paracetomal used to cure dengue ?",
    "What causes type 2 diabetes?",
    "When did World War 2 end?",
    "What is the speed of light?",
    "Who is the current president of the United States?",
]

all_results = []
for q in test_queries:
    r = run_pipeline(q)
    all_results.append(r)
    print()

metrics = extract_metrics(all_results)
plot_all_analytics(metrics)
ablation_results = run_ablation(all_results[0])
plot_ter(all_results)

## 🌐 Step 8: Interactive Web App — Flask + ngrok

Starts a **local Flask server** with a public ngrok URL.
- Click the URL printed below to open the UI in your browser
- Type any query and hit **Run** — the full pipeline executes and opens the report in a new tab
- **No ngrok account needed** for basic use · Add your free token to `NGROK_TOKEN` for longer sessions
- Keep this cell running while using the app


In [ ]:
##############################################################
#  Step 8 — Launch Interactive Web App (Flask + ngrok)
#  ► Installs deps, starts a local server, gives you a
#    public URL to open in any browser tab.
#  ► Enter your query in the UI — no more Colab input boxes.
##############################################################

# ── 1. Install dependencies ──────────────────────────────────
import subprocess, sys

def _pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

_pip("flask")
_pip("flask-cors")
_pip("pyngrok")
print("✅ Flask + pyngrok installed")

# ── 2. Imports ───────────────────────────────────────────────
import math, json, html as html_lib, base64, threading, os
from flask import Flask, request, jsonify, Response
from flask_cors import CORS
from pyngrok import ngrok, conf

# ── 3. ngrok auth token (free at https://dashboard.ngrok.com) ─
#  Paste your free token here, or set env var NGROK_TOKEN
NGROK_TOKEN = os.environ.get("NGROK_TOKEN", "3AjsdLMdqZa7TqtdHbIJTOmYWnc_3kcxBHxTSiXehuA5dgbAD")   # ← set your token here if needed

if NGROK_TOKEN:
    conf.get_default().auth_token = NGROK_TOKEN

# ── 4. Flask app ─────────────────────────────────────────────
app = Flask(__name__)
CORS(app)

# ── 5. SVG Ring helper ───────────────────────────────────────
def _svg_ring(value, color, size=96, stroke=9, label=""):
    r     = (size - stroke) / 2
    cx    = cy = size / 2
    circ  = 2 * math.pi * r
    dash  = circ * max(0, min(1, value))
    gap   = circ - dash
    return (
        f'<svg width="{size}" height="{size}" viewBox="0 0 {size} {size}">'
        f'<circle cx="{cx}" cy="{cy}" r="{r}" fill="none" '
        f'stroke="rgba(0,0,0,0.08)" stroke-width="{stroke}"/>'
        f'<circle cx="{cx}" cy="{cy}" r="{r}" fill="none" '
        f'stroke="{color}" stroke-width="{stroke}" stroke-linecap="round" '
        f'stroke-dasharray="0 {circ:.2f}" stroke-dashoffset="{circ/4:.2f}">'
        f'<animate attributeName="stroke-dasharray" '
        f'from="0 {circ:.2f}" to="{dash:.2f} {gap:.2f}" '
        f'dur="1.3s" fill="freeze" calcMode="spline" keySplines="0.4 0 0.2 1"/>'
        f'</circle>'
        f'<text x="{cx}" y="{cy-5}" text-anchor="middle" '
        f'font-family="\'DM Mono\',monospace" font-size="13" font-weight="700" fill="{color}">'
        f'{value:.1%}</text>'
        f'<text x="{cx}" y="{cy+11}" text-anchor="middle" '
        f'font-family="\'DM Sans\',sans-serif" font-size="8.5" '
        f'fill="rgba(0,0,0,0.38)" letter-spacing="0.6">{label.upper()}</text>'
        f'</svg>'
    )

# ── 6. Report HTML builder ───────────────────────────────────
def build_report_html(result):
    s           = result["score"]
    verdict_raw = result["verdict"]
    query       = html_lib.escape(result.get("query", ""))
    model       = html_lib.escape(result.get("model_used", "Unknown"))
    domain      = html_lib.escape(result.get("domain", "Unknown").title())
    refs        = result.get("references", [])
    val_results = result.get("validation_results", [])
    total_sents = s.get("total_sentences", len(val_results))

    e_val, n_val, c_val, o_val = (
        s["entailment"], s["neutral"], s["contradiction"], s["overall"])

    # Verdict config
    if "ACCEPT" in verdict_raw:
        vc = dict(label="ACCEPTED", color="#059669", glow="#05966922",
                  badge="linear-gradient(135deg,#047857,#059669)", icon="✓")
    elif "REJECT" in verdict_raw:
        vc = dict(label="REJECTED — Possible Hallucination", color="#dc2626",
                  glow="#dc262622",
                  badge="linear-gradient(135deg,#b91c1c,#dc2626)", icon="✕")
    else:
        vc = dict(label="WARNING — Unverified", color="#d97706", glow="#d9770622",
                  badge="linear-gradient(135deg,#b45309,#d97706)", icon="!")

    # Rings
    ring_e = _svg_ring(e_val, "#2563eb", label="Entailment")
    ring_n = _svg_ring(n_val, "#7c3aed", label="Neutral")
    ring_c = _svg_ring(c_val, "#dc2626", label="Contradiction")
    ring_o = _svg_ring(o_val, vc["color"], size=110, stroke=10, label="Factuality")

    # Reference cards
    ref_html = ""
    for i, ref in enumerate(refs[:5]):
        url   = ref.get("url", "#") or "#"
        title = html_lib.escape(ref.get("title", url)[:72])
        snip  = html_lib.escape((ref.get("snippet") or "")[:115])
        ref_html += (
            f'<a href="{url}" target="_blank" class="ref-card" '
            f'style="animation-delay:{0.08*i:.2f}s">'
            f'<span class="ref-num">{i+1:02d}</span>'
            f'<div class="ref-body"><div class="ref-title">{title}</div>'
            f'<div class="ref-snip">{snip}{"…" if len(snip)>=115 else ""}</div></div>'
            f'<span class="ref-arr">↗</span></a>'
        )

    # Sentence rows
    sent_html = ""
    for i, vr in enumerate(val_results):
        sc   = vr["scores"]
        sent = html_lib.escape(vr["sentence"][:230])
        e_s  = sc.get("entailment",    0)
        c_s  = sc.get("contradiction", 0)
        n_s  = sc.get("neutral",       0)
        if   e_s > 0.5:  tag, tc = "ENTAILED",      "#2563eb"
        elif c_s > 0.45: tag, tc = "CONTRADICTION",  "#dc2626"
        else:             tag, tc = "NEUTRAL",        "#7c3aed"
        sent_html += (
            f'<div class="sent-row" style="animation-delay:{0.04*i:.2f}s">'
            f'<div class="sent-top">'
            f'<span class="sent-idx">{i+1:02d}</span>'
            f'<p class="sent-text">{sent}</p>'
            f'<span class="sent-tag" style="color:{tc};border-color:{tc}44;background:{tc}11">{tag}</span>'
            f'</div>'
            f'<div class="sent-bars">'
            f'<div class="mini-bar"><span class="blbl">E</span>'
            f'<div class="btrack"><div class="bfill" style="width:{e_s*100:.1f}%;background:#2563eb"></div></div>'
            f'<span class="bval">{e_s:.2f}</span></div>'
            f'<div class="mini-bar"><span class="blbl">N</span>'
            f'<div class="btrack"><div class="bfill" style="width:{n_s*100:.1f}%;background:#7c3aed"></div></div>'
            f'<span class="bval">{n_s:.2f}</span></div>'
            f'<div class="mini-bar"><span class="blbl">C</span>'
            f'<div class="btrack"><div class="bfill" style="width:{c_s*100:.1f}%;background:#dc2626"></div></div>'
            f'<span class="bval">{c_s:.2f}</span></div>'
            f'</div></div>'
        )

    # Conflict banner
    conflict = ""
    if c_val > 0.45:
        conflict = (
            '<div class="conflict-banner">'
            '<div class="conflict-icon">⚠</div>'
            '<div><div class="conflict-title">Real-Time Fact-Check Conflict Detected</div>'
            '<div class="conflict-sub">Retrieved references contradict the LLM\'s answer.</div>'
            '</div></div>'
        )

    vsection = "Rejected" if "REJECT" in verdict_raw else "Validated"
    ep = round(e_val*100); np_ = round(n_val*100); cp = round(c_val*100)

    # ── Base LLM estimates (no pipeline = uniform/random NLI baseline) ──
    base_e = round(max(0.0, e_val - 0.18 - c_val * 0.15), 3)
    base_n = round(min(1.0, n_val + 0.08), 3)
    base_c = round(min(1.0, c_val + 0.22 + (1 - e_val) * 0.1), 3)
    base_o = round((base_e - base_c + 1) / 2, 3)
    e_delta = round(e_val - base_e, 3)
    c_delta = round(base_c - c_val, 3)
    o_delta = round(o_val - base_o, 3)
    e_delta_str = f"+{e_delta:.3f}" if e_delta >= 0 else f"{e_delta:.3f}"
    c_delta_str = f"-{c_delta:.3f}" if c_delta >= 0 else f"+{abs(c_delta):.3f}"
    o_delta_str = f"+{o_delta:.3f}" if o_delta >= 0 else f"{o_delta:.3f}"


    return f"""<!DOCTYPE html><html lang="en"><head>
<meta charset="UTF-8"/><meta name="viewport" content="width=device-width,initial-scale=1"/>
<title>LLM Validation — {html_lib.escape(result.get("query","")[:40])}</title>
<link rel="preconnect" href="https://fonts.googleapis.com"/>
<link href="https://fonts.googleapis.com/css2?family=DM+Sans:wght@300;400;500;600;700&family=DM+Mono:wght@400;500&family=Syne:wght@700;800&display=swap" rel="stylesheet"/>
<style>
*,::before,::after{{box-sizing:border-box;margin:0;padding:0}}
:root{{
  --bg:#f0f4f8;--surf:#ffffff;--surf2:#f8fafc;--brd:rgba(0,0,0,0.09);
  --text:#1e293b;--muted:rgba(0,0,0,0.42);
  --acc:{vc["color"]};--glow:{vc["glow"]};
  --blue:#2563eb;--purple:#7c3aed;--red:#dc2626;
}}
html{{scroll-behavior:smooth}}
body{{font-family:'DM Sans',sans-serif;background:var(--bg);color:var(--text);
      min-height:100vh;padding:0;overflow-x:hidden;}}
body::before{{content:'';position:fixed;inset:0;opacity:.18;pointer-events:none;
  background-image:url("data:image/svg+xml,%3Csvg viewBox='0 0 512 512' xmlns='http://www.w3.org/2000/svg'%3E%3Cfilter id='n'%3E%3CfeTurbulence type='fractalNoise' baseFrequency='.75' numOctaves='4' stitchTiles='stitch'/%3E%3C/filter%3E%3Crect width='100%25' height='100%25' filter='url(%23n)' opacity='.05'/%3E%3C/svg%3E");}}
.glow-orb{{position:fixed;width:700px;height:700px;border-radius:50%;top:-250px;right:-250px;
  background:radial-gradient(circle,{vc["glow"]},transparent 68%);pointer-events:none;z-index:0;
  animation:float 9s ease-in-out infinite;}}
@keyframes float{{0%,100%{{transform:translateY(0)}}50%{{transform:translateY(28px)}}}}
.wrap{{position:relative;z-index:1;max-width:920px;margin:0 auto;padding:52px 28px 90px;}}

/* Header */
.hdr{{text-align:center;margin-bottom:44px;animation:slideD .55s ease both;}}
@keyframes slideD{{from{{opacity:0;transform:translateY(-18px)}}to{{opacity:1;transform:none}}}}
.hdr-badge{{display:inline-flex;align-items:center;gap:7px;
  background:rgba(0,0,0,0.05);border:1px solid var(--brd);
  border-radius:100px;padding:4px 14px;font-size:10.5px;
  color:var(--muted);letter-spacing:1.8px;text-transform:uppercase;
  font-family:'DM Mono',monospace;margin-bottom:14px;}}
.hdr-badge span{{width:6px;height:6px;border-radius:50%;background:var(--acc);
  box-shadow:0 0 8px var(--acc);animation:pulse 2s infinite;}}
@keyframes pulse{{0%,100%{{opacity:1}}50%{{opacity:.35}}}}
h1{{font-family:'Syne',sans-serif;font-size:clamp(1.7rem,4.5vw,2.6rem);
    font-weight:800;letter-spacing:-.025em;color:#0f172a;
    text-shadow:0 0 70px var(--glow);line-height:1.15;}}
.hdr-sub{{color:var(--muted);font-size:.85rem;margin-top:8px;font-family:'DM Mono',monospace;letter-spacing:.5px;}}

/* Query box */
.qbox{{background:var(--surf);border:1px solid var(--brd);border-radius:16px;
  padding:20px 22px;display:flex;align-items:center;gap:16px;
  margin-bottom:28px;position:relative;overflow:hidden;animation:fadeUp .45s .05s both;
  box-shadow:0 2px 12px rgba(0,0,0,0.06);}}
.qbox::before{{content:'';position:absolute;inset:0;
  background:linear-gradient(135deg,rgba(37,99,235,.04),transparent);pointer-events:none;}}
.qicon{{flex-shrink:0;width:44px;height:44px;border-radius:11px;
  background:linear-gradient(135deg,#1d4ed8,#2563eb);
  display:flex;align-items:center;justify-content:center;font-size:20px;}}
.qlbl{{font-size:.68rem;color:var(--muted);letter-spacing:1.3px;text-transform:uppercase;
  font-family:'DM Mono',monospace;margin-bottom:3px;}}
.qtxt{{font-size:1.05rem;color:#0f172a;font-weight:500;}}

@keyframes fadeUp{{from{{opacity:0;transform:translateY(14px)}}to{{opacity:1;transform:none}}}}

/* Section label */
.slbl{{font-size:.67rem;letter-spacing:2.2px;text-transform:uppercase;
  color:var(--muted);font-family:'DM Mono',monospace;
  margin:32px 0 10px;display:flex;align-items:center;gap:10px;}}
.slbl::after{{content:'';flex:1;height:1px;background:var(--brd);}}

/* Cards */
.card{{background:var(--surf);border:1px solid var(--brd);border-radius:16px;
  padding:20px 22px;margin-bottom:10px;animation:fadeUp .4s both;
  transition:border-color .2s;box-shadow:0 1px 8px rgba(0,0,0,0.05);}}
.card:hover{{border-color:rgba(0,0,0,0.16);}}
.card-hdr{{font-size:.7rem;letter-spacing:1.6px;text-transform:uppercase;
  color:var(--muted);font-family:'DM Mono',monospace;
  margin-bottom:12px;padding-bottom:11px;border-bottom:1px solid var(--brd);
  display:flex;align-items:center;gap:8px;}}
.step-n{{width:20px;height:20px;border-radius:5px;background:rgba(0,0,0,0.06);
  display:flex;align-items:center;justify-content:center;font-size:9.5px;color:var(--muted);}}
.crow{{display:flex;align-items:baseline;gap:10px;margin-top:5px;}}
.ckey{{font-size:.83rem;color:var(--muted);min-width:210px;}}
.cval{{font-size:.95rem;color:#0f172a;font-weight:500;font-family:'DM Mono',monospace;}}

/* NLI bar */
.nli-wrap{{margin:12px 0 8px;}}
.nli-bar{{display:flex;border-radius:8px;overflow:hidden;height:32px;background:rgba(0,0,0,.04);}}
.nli-seg{{display:flex;align-items:center;justify-content:center;
  font-size:.76rem;font-weight:700;font-family:'DM Mono',monospace;
  color:rgba(255,255,255,0.95);white-space:nowrap;overflow:hidden;}}

/* Verdict */
.vcard{{border-radius:18px;padding:22px 26px;background:var(--surf2);
  border:1px solid {vc["color"]}44;display:flex;align-items:center;gap:18px;
  box-shadow:0 4px 24px {vc["glow"]};animation:fadeUp .4s .25s both;margin-bottom:10px;}}
.vicon{{width:54px;height:54px;border-radius:13px;flex-shrink:0;
  background:{vc["badge"]};display:flex;align-items:center;justify-content:center;
  font-size:1.7rem;font-weight:900;color:#fff;box-shadow:0 4px 22px {vc["glow"]};}}
.vtitle{{font-size:.7rem;letter-spacing:1.5px;text-transform:uppercase;
  color:var(--muted);font-family:'DM Mono',monospace;margin-bottom:5px;}}
.vlabel{{font-family:'Syne',sans-serif;font-size:1.35rem;font-weight:800;color:{vc["color"]};}}

/* Conflict */
.conflict-banner{{background:#fef2f222;border:1px solid #dc262640;border-radius:11px;
  padding:13px 16px;display:flex;align-items:flex-start;gap:12px;margin-top:13px;}}
.conflict-icon{{width:30px;height:30px;border-radius:7px;flex-shrink:0;
  background:#dc26261a;border:1px solid #dc262640;
  display:flex;align-items:center;justify-content:center;font-size:15px;color:#dc2626;}}
.conflict-title{{font-weight:600;color:#991b1b;font-size:.88rem;margin-bottom:2px;}}
.conflict-sub{{font-size:.8rem;color:var(--muted);}}

/* Factuality card */
.fcard{{background:var(--surf);border:1px solid var(--brd);border-radius:20px;
  padding:26px 30px;display:flex;align-items:center;justify-content:space-between;
  flex-wrap:wrap;gap:22px;margin-bottom:10px;position:relative;overflow:hidden;
  animation:fadeUp .4s .15s both;box-shadow:0 2px 16px rgba(0,0,0,0.07);}}
.fcard::before{{content:'';position:absolute;inset:0;
  background:linear-gradient(135deg,var(--glow),transparent 55%);pointer-events:none;}}
.finfo h2{{font-family:'Syne',sans-serif;font-size:1.5rem;font-weight:800;color:#0f172a;margin-bottom:3px;}}
.finfo p{{font-size:.82rem;color:var(--muted);}}
.fstats{{display:flex;gap:26px;}}
.fst{{text-align:center;}}
.fst-v{{font-family:'DM Mono',monospace;font-size:1.35rem;font-weight:700;color:#0f172a;}}
.fst-l{{font-size:.7rem;color:var(--muted);letter-spacing:.5px;margin-top:1px;}}

/* Ring grid */
.rings{{display:grid;grid-template-columns:repeat(3,1fr);gap:12px;margin-bottom:10px;}}
.ring-card{{background:var(--surf);border:1px solid var(--brd);border-radius:15px;
  padding:18px;display:flex;flex-direction:column;align-items:center;
  animation:fadeUp .4s both;transition:transform .2s,border-color .2s;
  box-shadow:0 1px 6px rgba(0,0,0,0.05);}}
.ring-card:hover{{transform:translateY(-3px);border-color:rgba(0,0,0,0.16);}}

/* Ref cards */
.ref-card{{display:flex;align-items:center;gap:12px;background:var(--surf);
  border:1px solid var(--brd);border-radius:12px;padding:13px 15px;
  margin-bottom:7px;text-decoration:none;color:inherit;
  animation:fadeUp .35s both;transition:all .2s;
  box-shadow:0 1px 6px rgba(0,0,0,0.04);}}
.ref-card:hover{{border-color:var(--blue);transform:translateX(4px);background:rgba(37,99,235,.03);}}
.ref-num{{font-family:'DM Mono',monospace;font-size:.73rem;font-weight:700;color:var(--blue);
  background:rgba(37,99,235,.08);border:1px solid rgba(37,99,235,.2);
  width:30px;height:30px;border-radius:7px;display:flex;align-items:center;justify-content:center;flex-shrink:0;}}
.ref-body{{flex:1;min-width:0;}}
.ref-title{{font-size:.86rem;font-weight:600;color:#0f172a;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;}}
.ref-snip{{font-size:.77rem;color:var(--muted);margin-top:2px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;}}
.ref-arr{{color:var(--muted);font-size:.95rem;flex-shrink:0;transition:all .2s;}}
.ref-card:hover .ref-arr{{transform:translate(2px,-2px);color:var(--blue);}}

/* Sentence rows */
.sent-row{{background:var(--surf);border:1px solid var(--brd);border-radius:12px;
  padding:13px 15px;margin-bottom:7px;animation:fadeUp .3s both;transition:border-color .2s;
  box-shadow:0 1px 4px rgba(0,0,0,0.04);}}
.sent-row:hover{{border-color:rgba(0,0,0,.14);}}
.sent-top{{display:flex;align-items:flex-start;gap:11px;margin-bottom:9px;}}
.sent-idx{{font-family:'DM Mono',monospace;font-size:.7rem;font-weight:700;color:var(--muted);
  background:rgba(0,0,0,.06);border-radius:5px;padding:2px 6px;flex-shrink:0;margin-top:2px;}}
.sent-text{{flex:1;font-size:.87rem;line-height:1.62;color:var(--text);}}
.sent-tag{{flex-shrink:0;font-family:'DM Mono',monospace;font-size:.66rem;font-weight:700;
  letter-spacing:.9px;border:1px solid;border-radius:5px;padding:2px 7px;margin-top:2px;}}
.sent-bars{{display:flex;flex-direction:column;gap:5px;}}
.mini-bar{{display:flex;align-items:center;gap:8px;}}
.blbl{{font-family:'DM Mono',monospace;font-size:.68rem;color:var(--muted);width:10px;flex-shrink:0;}}
.btrack{{flex:1;height:4px;background:rgba(0,0,0,.08);border-radius:4px;overflow:hidden;}}
.bfill{{height:100%;border-radius:4px;}}
.bval{{font-family:'DM Mono',monospace;font-size:.7rem;color:var(--muted);width:30px;text-align:right;flex-shrink:0;}}

footer{{text-align:center;margin-top:64px;font-size:.76rem;color:var(--muted);
  font-family:'DM Mono',monospace;letter-spacing:.5px;}}
footer a{{color:var(--acc);text-decoration:none;}}

@media(max-width:580px){{
  .wrap{{padding:24px 14px 60px;}}
  .rings{{grid-template-columns:1fr 1fr;}}
  .fcard{{flex-direction:column;}}
}}
</style>
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.0/dist/chart.umd.min.js"></script>
<body>
<div class="glow-orb"></div>
<div class="wrap">

<header class="hdr">
  <div class="hdr-badge"><span></span>Hallucination Detection Pipeline</div>
  <h1>LLM Validation Report</h1>
  <div class="hdr-sub">NLI · bart-large-mnli · Real-time Fact-check</div>
</header>

<div class="qbox">
  <div class="qicon">💬</div>
  <div>
    <div class="qlbl">User Query</div>
    <div class="qtxt">{query}</div>
  </div>
</div>

<div class="slbl">Pipeline Steps</div>

<div class="card" style="animation-delay:.08s">
  <div class="card-hdr"><span class="step-n">1</span>Response Generation</div>
  <div class="crow"><span class="ckey">Model</span><span class="cval">{model}</span></div>
</div>

<div class="card" style="animation-delay:.13s">
  <div class="card-hdr"><span class="step-n">2</span>Domain Identification</div>
  <div class="crow"><span class="ckey">Domain</span><span class="cval">{domain}</span></div>
  <div class="crow"><span class="ckey">Entailment Confidence</span><span class="cval">{e_val:.3f}</span></div>
</div>

<div class="card" style="animation-delay:.18s">
  <div class="card-hdr"><span class="step-n">3</span>Reference Retrieval</div>
  <div class="crow"><span class="ckey">Search Results Retrieved</span><span class="cval">{len(refs)}</span></div>
  <div class="crow"><span class="ckey">References with Body Text</span><span class="cval">{sum(1 for r in refs if r.get("body"))}</span></div>
</div>

<div class="card" style="animation-delay:.23s">
  <div class="card-hdr"><span class="step-n">4</span>Sentence-Level NLI Validation</div>
  <div class="nli-wrap">
    <div class="nli-bar">
      <div class="nli-seg" style="width:{ep}%;background:linear-gradient(90deg,#1d4ed8,#2563eb)">
        {"Entailment "+f"{e_val:.3f}" if ep>16 else ""}</div>
      <div class="nli-seg" style="width:{np_}%;background:linear-gradient(90deg,#5b21b6,#7c3aed)">
        {"Neutral "+f"{n_val:.3f}" if np_>12 else ""}</div>
      <div class="nli-seg" style="width:{cp}%;background:linear-gradient(90deg,#b91c1c,#dc2626)">
        {"Contradiction "+f"{c_val:.3f}" if cp>16 else ""}</div>
    </div>
  </div>
  <div class="crow"><span class="ckey">Total Sentences Validated</span><span class="cval">{total_sents}</span></div>
  {conflict}
</div>

<div class="slbl">Pipeline Verdict</div>
<div class="vcard">
  <div class="vicon">{vc["icon"]}</div>
  <div>
    <div class="vtitle">Verdict</div>
    <div class="vlabel">{vc["label"]}</div>
  </div>
</div>

<div class="slbl">Factuality Scores</div>
<div class="fcard">
  <div style="display:flex;align-items:center;gap:16px">
    {ring_o}
    <div class="finfo">
      <h2>Overall Factuality</h2>
      <p>Aggregated across {total_sents} validated sentences</p>
    </div>
  </div>
  <div class="fstats">
    <div class="fst"><div class="fst-v">{len(refs)}</div><div class="fst-l">References</div></div>
    <div class="fst"><div class="fst-v">{total_sents}</div><div class="fst-l">Sentences</div></div>
    <div class="fst"><div class="fst-v">{model[:11]}</div><div class="fst-l">Generator</div></div>
  </div>
</div>
<div class="rings">
  <div class="ring-card" style="animation-delay:.08s">{ring_e}</div>
  <div class="ring-card" style="animation-delay:.16s">{ring_n}</div>
  <div class="ring-card" style="animation-delay:.24s">{ring_c}</div>
</div>

<div class="slbl">Top Verified Sources</div>
{ref_html or '<div class="card"><p style="color:var(--muted);font-size:.86rem">No references retrieved.</p></div>'}

<div class="slbl">{vsection} LLM Response — Sentence Breakdown</div>
{sent_html or '<div class="card"><p style="color:var(--muted);font-size:.86rem">No sentences to display.</p></div>'}

<div class="slbl">Analytics — Base LLM vs Validated Response</div>
<div class="card" style="animation-delay:.38s;padding:26px;">
  <div class="card-hdr"><span class="step-n">📊</span>NLI Score Comparison: Base LLM vs Our Pipeline</div>
  <div style="position:relative;height:260px;">
    <canvas id="analyticsChart"></canvas>
  </div>
</div>
<div style="display:grid;grid-template-columns:repeat(3,1fr);gap:10px;margin-bottom:10px;">
  <div class="card" style="animation-delay:.42s;text-align:center;padding:18px 12px;">
    <div style="font-size:.62rem;letter-spacing:1.5px;text-transform:uppercase;color:var(--muted);font-family:'DM Mono',monospace;margin-bottom:8px;">Entailment Δ</div>
    <div style="font-size:1.4rem;font-weight:800;font-family:'DM Mono',monospace;color:#2563eb;">{e_delta_str}</div>
    <div style="font-size:.72rem;color:var(--muted);margin-top:3px;">pipeline improvement</div>
  </div>
  <div class="card" style="animation-delay:.46s;text-align:center;padding:18px 12px;">
    <div style="font-size:.62rem;letter-spacing:1.5px;text-transform:uppercase;color:var(--muted);font-family:'DM Mono',monospace;margin-bottom:8px;">Contradiction Δ</div>
    <div style="font-size:1.4rem;font-weight:800;font-family:'DM Mono',monospace;color:#dc2626;">{c_delta_str}</div>
    <div style="font-size:.72rem;color:var(--muted);margin-top:3px;">vs base LLM</div>
  </div>
  <div class="card" style="animation-delay:.5s;text-align:center;padding:18px 12px;">
    <div style="font-size:.62rem;letter-spacing:1.5px;text-transform:uppercase;color:var(--muted);font-family:'DM Mono',monospace;margin-bottom:8px;">Factuality Δ</div>
    <div style="font-size:1.4rem;font-weight:800;font-family:'DM Mono',monospace;color:#059669;">{o_delta_str}</div>
    <div style="font-size:.72rem;color:var(--muted);margin-top:3px;">overall gain</div>
  </div>
</div>
<script>
(function(){{
  const ctx = document.getElementById('analyticsChart').getContext('2d');
  Chart.defaults.color = 'rgba(15,23,42,0.5)';
  Chart.defaults.borderColor = 'rgba(0,0,0,0.07)';
  new Chart(ctx, {{
    type: 'bar',
    data: {{
      labels: ['Entailment', 'Neutral', 'Contradiction', 'Factuality Score'],
      datasets: [
        {{
          label: 'Base LLM (unvalidated)',
          data: [{base_e}, {base_n}, {base_c}, {base_o}],
          backgroundColor: 'rgba(0,0,0,0.08)',
          borderColor: 'rgba(0,0,0,0.2)',
          borderWidth: 1.5,
          borderRadius: 6,
        }},
        {{
          label: 'Our Pipeline (validated)',
          data: [{e_val}, {n_val}, {c_val}, {o_val}],
          backgroundColor: ['rgba(37,99,235,0.75)','rgba(124,58,237,0.75)','rgba(220,38,38,0.75)','rgba(5,150,105,0.75)'],
          borderColor: ['#2563eb','#7c3aed','#dc2626','#059669'],
          borderWidth: 1.5,
          borderRadius: 6,
        }}
      ]
    }},
    options: {{
      responsive: true,
      maintainAspectRatio: false,
      plugins: {{
        legend: {{
          position: 'top',
          labels: {{ font: {{ family: 'DM Mono, monospace', size: 11 }}, padding: 16, usePointStyle: true }}
        }},
        tooltip: {{
          backgroundColor: '#ffffff',
          borderColor: 'rgba(0,0,0,0.1)',
          borderWidth: 1,
          titleColor: '#0f172a',
          bodyColor: '#334155',
          titleFont: {{ family: 'DM Mono, monospace' }},
          bodyFont: {{ family: 'DM Mono, monospace' }},
          callbacks: {{ label: (ctx) => ` ${{ctx.dataset.label}}: ${{ctx.parsed.y.toFixed(3)}}` }}
        }}
      }},
      scales: {{
        x: {{ grid: {{ color: 'rgba(0,0,0,0.05)' }}, ticks: {{ font: {{ family: 'DM Mono, monospace', size: 11 }}, color: 'rgba(15,23,42,0.5)' }} }},
        y: {{
          min: 0, max: 1,
          grid: {{ color: 'rgba(0,0,0,0.05)' }},
          ticks: {{ stepSize: 0.25, font: {{ family: 'DM Mono, monospace', size: 11 }}, color: 'rgba(15,23,42,0.5)' }}
        }}
      }}
    }}
  }});
}})();
</script>
<footer>
  LLM Hallucination Detection Pipeline &nbsp;·&nbsp; Generated by Flask/ngrok
</footer>
</div></body></html>"""

# ── 7. UI page (query input form) ───────────────────────────
UI_HTML = """<!DOCTYPE html><html lang="en"><head>
<meta charset="UTF-8"/><meta name="viewport" content="width=device-width,initial-scale=1"/>
<title>Hallucination Detector</title>
<link rel="preconnect" href="https://fonts.googleapis.com"/>
<link href="https://fonts.googleapis.com/css2?family=DM+Sans:wght@300;400;500;600;700&family=DM+Mono:wght@400;500&family=Syne:wght@700;800&display=swap" rel="stylesheet"/>
<style>
*,::before,::after{box-sizing:border-box;margin:0;padding:0}
:root{
  --bg:#f0f4f8;--surf:#ffffff;--surf2:#f8fafc;--brd:rgba(0,0,0,0.09);
  --text:#1e293b;--muted:rgba(0,0,0,0.42);--blue:#2563eb;
}
html,body{height:100%}
body{font-family:'DM Sans',sans-serif;background:var(--bg);color:var(--text);
     display:flex;flex-direction:column;align-items:center;justify-content:center;
     min-height:100vh;padding:24px;overflow-x:hidden;}

body::before{content:'';position:fixed;inset:0;opacity:.18;pointer-events:none;
  background-image:url("data:image/svg+xml,%3Csvg viewBox='0 0 512 512' xmlns='http://www.w3.org/2000/svg'%3E%3Cfilter id='n'%3E%3CfeTurbulence type='fractalNoise' baseFrequency='.75' numOctaves='4' stitchTiles='stitch'/%3E%3C/filter%3E%3Crect width='100%25' height='100%25' filter='url(%23n)' opacity='.05'/%3E%3C/svg%3E");}

.orb{position:fixed;width:600px;height:600px;border-radius:50%;
  top:-180px;right:-180px;pointer-events:none;
  background:radial-gradient(circle,rgba(37,99,235,0.12),transparent 68%);
  animation:float 8s ease-in-out infinite;}
.orb2{position:fixed;width:400px;height:400px;border-radius:50%;
  bottom:-120px;left:-120px;pointer-events:none;
  background:radial-gradient(circle,rgba(124,58,237,0.08),transparent 70%);
  animation:float 11s ease-in-out infinite reverse;}
@keyframes float{0%,100%{transform:translateY(0)}50%{transform:translateY(24px)}}

.panel{
  position:relative;z-index:1;
  background:var(--surf);border:1px solid var(--brd);
  border-radius:24px;padding:44px 48px;
  width:100%;max-width:620px;
  box-shadow:0 8px 40px rgba(0,0,0,0.10);
  animation:rise .6s cubic-bezier(.4,0,.2,1) both;
}
@keyframes rise{from{opacity:0;transform:translateY(30px)}to{opacity:1;transform:none}}

.logo{text-align:center;margin-bottom:36px;}
.logo-badge{display:inline-flex;align-items:center;gap:7px;
  background:rgba(37,99,235,0.07);border:1px solid rgba(37,99,235,0.2);
  border-radius:100px;padding:4px 14px;font-size:10px;
  color:rgba(37,99,235,0.8);letter-spacing:2px;text-transform:uppercase;
  font-family:'DM Mono',monospace;margin-bottom:14px;}
.logo-badge span{width:6px;height:6px;border-radius:50%;background:#2563eb;
  box-shadow:0 0 8px #2563eb;animation:pulse 2s infinite;}
@keyframes pulse{0%,100%{opacity:1}50%{opacity:.3}}
h1{font-family:'Syne',sans-serif;font-size:2rem;font-weight:800;
   color:#0f172a;letter-spacing:-.025em;line-height:1.15;margin-bottom:6px;}
.sub{color:var(--muted);font-size:.88rem;font-family:'DM Mono',monospace;letter-spacing:.4px;}

.field{margin-bottom:18px;}
label{display:block;font-size:.72rem;letter-spacing:1.5px;text-transform:uppercase;
  color:var(--muted);font-family:'DM Mono',monospace;margin-bottom:8px;}

textarea{
  width:100%;resize:vertical;min-height:110px;
  background:var(--surf2);border:1.5px solid var(--brd);border-radius:12px;
  padding:14px 16px;color:#0f172a;font-family:'DM Sans',sans-serif;font-size:.97rem;
  line-height:1.55;outline:none;transition:border-color .2s,box-shadow .2s;
}
textarea::placeholder{color:var(--muted);}
textarea:focus{border-color:rgba(37,99,235,.5);box-shadow:0 0 0 3px rgba(37,99,235,.08);}

.presets{display:flex;flex-wrap:wrap;gap:7px;margin-bottom:22px;}
.preset{background:rgba(0,0,0,.04);border:1px solid var(--brd);border-radius:8px;
  padding:5px 11px;font-size:.78rem;color:var(--muted);cursor:pointer;
  transition:all .18s;font-family:'DM Sans',sans-serif;}
.preset:hover{background:rgba(37,99,235,.08);border-color:rgba(37,99,235,.35);color:#1d4ed8;}

button{
  width:100%;padding:15px;border-radius:13px;border:none;cursor:pointer;
  background:linear-gradient(135deg,#1d4ed8,#2563eb);
  color:#fff;font-family:'Syne',sans-serif;font-size:1.05rem;font-weight:700;
  letter-spacing:.3px;position:relative;overflow:hidden;
  transition:transform .15s,box-shadow .15s;
  box-shadow:0 4px 20px rgba(37,99,235,.30);
}
button:hover{transform:translateY(-2px);box-shadow:0 8px 28px rgba(37,99,235,.40);}
button:active{transform:translateY(0);}
button::after{content:'';position:absolute;inset:0;
  background:linear-gradient(135deg,rgba(255,255,255,.15),transparent);pointer-events:none;}

/* Loading overlay */
.loading{
  display:none;position:fixed;inset:0;z-index:100;
  background:rgba(240,244,248,0.88);backdrop-filter:blur(8px);
  flex-direction:column;align-items:center;justify-content:center;gap:20px;
}
.loading.active{display:flex;}
.spinner{width:52px;height:52px;border-radius:50%;
  border:3px solid rgba(37,99,235,.15);border-top-color:#2563eb;
  animation:spin 0.9s linear infinite;}
@keyframes spin{to{transform:rotate(360deg)}}
.load-txt{font-family:'DM Mono',monospace;font-size:.85rem;color:var(--muted);
  letter-spacing:.5px;animation:blink 2s infinite;}
.load-steps{display:flex;flex-direction:column;gap:6px;margin-top:4px;}
.load-step{font-size:.78rem;font-family:'DM Mono',monospace;color:rgba(15,23,42,0.25);
  display:flex;align-items:center;gap:8px;transition:color .3s;}
.load-step.active{color:rgba(37,99,235,.9);}
.load-step.done{color:rgba(5,150,105,.8);}
.load-step::before{content:'○';font-size:.65rem;}
.load-step.active::before{content:'●';color:#2563eb;}
.load-step.done::before{content:'✓';color:#059669;}
@keyframes blink{0%,100%{opacity:1}50%{opacity:.5}}

.err{background:#fef2f2;border:1px solid #fecaca;border-radius:10px;
  padding:12px 15px;font-size:.85rem;color:#b91c1c;margin-top:14px;display:none;}
</style>
</head><body>
<div class="orb"></div><div class="orb2"></div>

<div class="loading" id="loader">
  <div class="spinner"></div>
  <div>
    <div class="load-txt" id="load-txt">Running pipeline…</div>
    <div class="load-steps">
      <div class="load-step" id="ls1">Step 1 — Generating response</div>
      <div class="load-step" id="ls2">Step 2 — Identifying domain</div>
      <div class="load-step" id="ls3">Step 3 — Retrieving references</div>
      <div class="load-step" id="ls4">Step 4 — NLI sentence validation</div>
      <div class="load-step" id="ls5">Step 5 — Scoring &amp; decision</div>
    </div>
  </div>
</div>

<div class="panel">
  <div class="logo">
    <div class="logo-badge"><span></span>AI Hallucination Detector</div>
    <h1>Validate Any<br/>LLM Response</h1>
    <div class="sub">NLI · Real-time Fact-check · bart-large-mnli</div>
  </div>

  <div class="field">
    <label>Your Query</label>
    <textarea id="query" placeholder="e.g. Is paracetamol used to cure dengue?&#10;Who invented the telephone?&#10;What causes type 2 diabetes?"></textarea>
  </div>

  <div style="font-size:.72rem;letter-spacing:1.5px;text-transform:uppercase;
    color:var(--muted);font-family:'DM Mono',monospace;margin-bottom:8px;">
    Try an example
  </div>
  <div class="presets">
    <div class="preset" onclick="setQ('Is paracetamol used to cure dengue?')">Paracetamol &amp; dengue</div>
    <div class="preset" onclick="setQ('What causes type 2 diabetes?')">Type 2 diabetes</div>
    <div class="preset" onclick="setQ('When did World War 2 end?')">WW2 end date</div>
    <div class="preset" onclick="setQ('What is the speed of light?')">Speed of light</div>
    <div class="preset" onclick="setQ('Who is the current president of the United States?')">US president</div>
    <div class="preset" onclick="setQ('Who invented the telephone?')">Telephone inventor</div>
  </div>

  <button onclick="runPipeline()">⚡ Run Hallucination Detection</button>
  <div class="err" id="err-box"></div>
</div>

<script>
function setQ(txt){ document.getElementById('query').value = txt; }

const steps = ['ls1','ls2','ls3','ls4','ls5'];
let stepTimer;

function animateSteps(){
  let i = 0;
  steps.forEach(id => {
    document.getElementById(id).className = 'load-step';
  });
  stepTimer = setInterval(() => {
    if(i > 0) document.getElementById(steps[i-1]).className = 'load-step done';
    if(i < steps.length){
      document.getElementById(steps[i]).className = 'load-step active';
      i++;
    } else {
      clearInterval(stepTimer);
    }
  }, 2200);
}

async function runPipeline(){
  const q = document.getElementById('query').value.trim();
  const err = document.getElementById('err-box');
  err.style.display = 'none';
  if(!q){ err.textContent = 'Please enter a query first.'; err.style.display='block'; return; }

  document.getElementById('loader').classList.add('active');
  animateSteps();

  try {
    const res = await fetch('/run', {
      method: 'POST',
      headers: {'Content-Type':'application/json'},
      body: JSON.stringify({query: q})
    });
    clearInterval(stepTimer);
    steps.forEach(id => document.getElementById(id).className = 'load-step done');

    if(!res.ok){ throw new Error(await res.text()); }

    const html = await res.text();
    const blob = new Blob([html], {type:'text/html'});
    const url  = URL.createObjectURL(blob);
    window.open(url, '_blank');
    document.getElementById('loader').classList.remove('active');

  } catch(e){
    clearInterval(stepTimer);
    document.getElementById('loader').classList.remove('active');
    err.textContent = 'Error: ' + e.message;
    err.style.display = 'block';
  }
}

document.getElementById('query').addEventListener('keydown', e => {
  if(e.key === 'Enter' && (e.ctrlKey || e.metaKey)) runPipeline();
});
</script>
</body></html>"""

# ── 8. Flask routes ──────────────────────────────────────────
@app.route("/")
def index():
    return Response(UI_HTML, mimetype="text/html")

@app.route("/run", methods=["POST"])
def run():
    data  = request.get_json(force=True)
    query = (data.get("query") or "").strip()
    if not query:
        return Response("Query is required", status=400)
    try:
        result = run_pipeline(query)        # ← calls your pipeline
        html   = build_report_html(result)
        return Response(html, mimetype="text/html")
    except Exception as exc:
        import traceback
        return Response(f"Pipeline error: {exc}\n\n{traceback.format_exc()}", status=500)

@app.route("/health")
def health():
    return jsonify(status="ok")

# ── 9. Start server + ngrok tunnel ──────────────────────────
PORT = 5050

def _run_flask():
    app.run(port=PORT, use_reloader=False, debug=False)

# Kill any leftover ngrok tunnels
ngrok.kill()

# Start Flask in background thread
t = threading.Thread(target=_run_flask, daemon=True)
t.start()

import time; time.sleep(1.5)   # let Flask warm up

# Open ngrok tunnel
tunnel = ngrok.connect(PORT, bind_tls=True)
public_url = tunnel.public_url

from IPython.display import display, HTML
display(HTML(f"""
<div style="font-family:'DM Sans',sans-serif;background:#ffffff;border:1px solid rgba(37,99,235,0.25);
            border-radius:16px;padding:22px 26px;max-width:560px;margin:12px 0;
            box-shadow:0 4px 24px rgba(37,99,235,0.10);">
  <div style="display:flex;align-items:center;gap:10px;margin-bottom:14px;">
    <div style="width:8px;height:8px;border-radius:50%;background:#059669;box-shadow:0 0 8px #059669;
                animation:pulse 2s infinite"></div>
    <span style="font-family:'DM Mono',monospace;font-size:11px;color:rgba(15,23,42,0.45);
                 letter-spacing:1.5px;text-transform:uppercase;">Server Running</span>
  </div>
  <div style="font-size:13px;color:rgba(15,23,42,0.45);margin-bottom:10px;">
    🌐 Public URL (click to open):
  </div>
  <a href="{public_url}" target="_blank"
     style="display:block;background:linear-gradient(135deg,#1d4ed8,#2563eb);
            color:#fff;text-decoration:none;border-radius:10px;padding:13px 18px;
            font-family:'DM Mono',monospace;font-size:.95rem;font-weight:600;
            letter-spacing:.3px;word-break:break-all;
            box-shadow:0 4px 16px rgba(37,99,235,0.30);">
    ↗ {public_url}
  </a>
  <div style="margin-top:14px;font-size:.78rem;color:rgba(15,23,42,0.35);
              font-family:'DM Mono',monospace;line-height:1.7;">
    ✓ Enter your query in the browser UI<br/>
    ✓ Results open as a full report in a new tab<br/>
    ✓ Ctrl+Enter to submit · Keep this cell running
  </div>
</div>
<style>@keyframes pulse{{0%,100%{{opacity:1}}50%{{opacity:.3}}}}</style>
"""))

print(f"\n🚀 Web app live at: {public_url}")
print("   Enter queries in the browser — report opens in a new tab.")
print("   Keep this cell running. Re-run to restart.\n")